# Install & imports

In [1]:
!pip install -q mne

import mne
import numpy as np
import pandas as pd
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

print('MNE version:', mne.__version__)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.7/7.7 MB 19.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
moviepy 1.0.3 requires decorator<5.0,>=4.0.2, but you have decorator 5.3.1 which is incompatible.
MNE version: 1.13.2


# Reconnect paths

In [2]:
from google.colab import drive
drive.mount('/content/drive')

PROJECT_ROOT = Path('/content/drive/MyDrive/PD_EEG_Thesis_Features_v2')
EDF_DIR      = PROJECT_ROOT / 'data' / 'edf'
SEG_DIR      = PROJECT_ROOT / 'data' / 'segments'
SEG_DIR.mkdir(parents=True, exist_ok=True)

manifest = pd.read_csv(PROJECT_ROOT / 'manifest.csv')
print(f'Subjects: {len(manifest)}')
print(manifest['group'].value_counts())

Mounted at /content/drive
Subjects: 149
group
PD         100
Control     49
Name: count, dtype: int64


# Segmentation parameters

In [3]:
WINDOW_SEC     = 6
SFREQ          = 500
WINDOW_SAMPLES = WINDOW_SEC * SFREQ   # 3000

LABEL_MAP = {'PD': 1, 'Control': 0}

print(f'Window: {WINDOW_SEC}s → {WINDOW_SAMPLES} samples per segment')

Window: 6s → 3000 samples per segment


# Canonical channel pre-scan

In [4]:
print('Scanning all subjects for canonical channel set...')
common_chs = None
for _, row in manifest.iterrows():
    raw = mne.io.read_raw_edf(row['edf_path'], preload=False, verbose=False)
    chs = set(raw.ch_names)
    common_chs = chs if common_chs is None else common_chs & chs

ref_raw = mne.io.read_raw_edf(manifest.iloc[0]['edf_path'], preload=False, verbose=False)
CANONICAL_CHANNELS = [ch for ch in ref_raw.ch_names if ch in common_chs]

print(f'Canonical channel set: {len(CANONICAL_CHANNELS)} channels')
print(CANONICAL_CHANNELS)

Scanning all subjects for canonical channel set...
Canonical channel set: 60 channels
['Fp1', 'Fz', 'F3', 'F7', 'FC5', 'FC1', 'C3', 'T7', 'TP9', 'CP5', 'CP1', 'P3', 'P7', 'O1', 'Oz', 'O2', 'P4', 'P8', 'TP10', 'CP6', 'CP2', 'Cz', 'C4', 'T8', 'FT10', 'FC6', 'FC2', 'F4', 'F8', 'Fp2', 'AF7', 'AF3', 'AFz', 'F1', 'F5', 'FT7', 'FC3', 'C1', 'C5', 'TP7', 'CP3', 'P1', 'P5', 'PO7', 'POz', 'PO8', 'P6', 'P2', 'CPz', 'CP4', 'TP8', 'C6', 'C2', 'FC4', 'FT8', 'F6', 'AF8', 'AF4', 'F2', 'FCz']


# Segmentation function

In [5]:
def segment_subject(edf_path, canonical_channels):
    raw = mne.io.read_raw_edf(edf_path, preload=True, verbose=False)
    picks = [ch for ch in canonical_channels if ch in raw.ch_names]
    raw_picked = raw.copy().pick(picks)
    data = raw_picked.get_data()

    # Handle any boundary annotations (unlikely in .edf exports, but check just in case)
    boundary_onsets = [ann['onset'] for ann in raw.annotations
                       if 'boundary' in ann['description'].lower()]
    total_dur = raw.times[-1]
    boundaries = sorted([0.0] + boundary_onsets + [total_dur])
    segments = [(boundaries[i], boundaries[i+1]) for i in range(len(boundaries)-1)]

    windows = []
    for seg_start, seg_end in segments:
        if (seg_end - seg_start) < WINDOW_SEC:
            continue
        s_start, s_end = int(seg_start * SFREQ), int(seg_end * SFREQ)
        seg_data = data[:, s_start:s_end]
        n_windows = seg_data.shape[1] // WINDOW_SAMPLES
        for w in range(n_windows):
            win = seg_data[:, w * WINDOW_SAMPLES : (w + 1) * WINDOW_SAMPLES]
            if win.shape[1] == WINDOW_SAMPLES:
                windows.append(win)

    return np.stack(windows, axis=0) if windows else None

print('Segmentation function defined.')

Segmentation function defined.


# Run segmentation for all subjects

In [6]:
X_list, y_list, subj_list = [], [], []

for _, row in manifest.iterrows():
    subj  = str(row['subject'])
    group = row['group']
    label = LABEL_MAP[group]

    windows = segment_subject(row['edf_path'], CANONICAL_CHANNELS)
    if windows is None:
        print(f'[!] sub-{subj}: no usable windows')
        continue

    X_list.append(windows)
    y_list.extend([label] * windows.shape[0])
    subj_list.extend([subj] * windows.shape[0])
    print(f'[✓] sub-{subj} ({group}): {windows.shape[0]} windows')

X = np.concatenate(X_list, axis=0)
y = np.array(y_list)
subjects = np.array(subj_list)

print(f'\nTotal: {X.shape}  PD={int(y.sum())}  Control={int((y==0).sum())}')

[✓] sub-1 (PD): 46 windows
[✓] sub-2 (PD): 54 windows
[✓] sub-3 (PD): 42 windows
[✓] sub-4 (PD): 43 windows
[✓] sub-5 (PD): 41 windows
[✓] sub-6 (PD): 43 windows
[✓] sub-7 (PD): 39 windows
[✓] sub-8 (PD): 39 windows
[✓] sub-9 (PD): 41 windows
[✓] sub-10 (PD): 57 windows
[✓] sub-11 (PD): 26 windows
[✓] sub-12 (PD): 20 windows
[✓] sub-13 (PD): 20 windows
[✓] sub-14 (PD): 20 windows
[✓] sub-15 (PD): 20 windows
[✓] sub-16 (PD): 20 windows
[✓] sub-17 (PD): 30 windows
[✓] sub-18 (PD): 25 windows
[✓] sub-19 (PD): 31 windows
[✓] sub-20 (PD): 30 windows
[✓] sub-21 (PD): 26 windows
[✓] sub-22 (PD): 25 windows
[✓] sub-23 (PD): 28 windows
[✓] sub-24 (PD): 25 windows
[✓] sub-25 (PD): 27 windows
[✓] sub-26 (PD): 23 windows
[✓] sub-27 (PD): 20 windows
[✓] sub-28 (PD): 25 windows
[✓] sub-29 (PD): 33 windows
[✓] sub-30 (PD): 29 windows
[✓] sub-31 (PD): 30 windows
[✓] sub-32 (PD): 25 windows
[✓] sub-33 (PD): 21 windows
[✓] sub-34 (PD): 29 windows
[✓] sub-35 (PD): 33 windows
[✓] sub-36 (PD): 20 windows
[

# Save

In [7]:
np.savez_compressed(str(SEG_DIR / 'segments_all.npz'), X=X, y=y, subjects=subjects)
print(f'Saved → segments_all.npz  |  Shape: {X.shape}')

Saved → segments_all.npz  |  Shape: (3915, 60, 3000)


# Verify

In [8]:
# Clip extreme artifact values per channel, based on percentiles across the
# whole dataset. This is standard practice for residual EEG artifacts that
# survive ICA — affects only 0.9% of windows, concentrated in 9 subjects.
print(f'Before clipping: min={X.min():.6f}  max={X.max():.6f}')

X_clipped = X.copy()
for ch_idx in range(X.shape[1]):
    p1, p99 = np.percentile(X[:, ch_idx, :], [0.5, 99.5])
    X_clipped[:, ch_idx, :] = np.clip(X[:, ch_idx, :], p1, p99)

print(f'After clipping : min={X_clipped.min():.6f}  max={X_clipped.max():.6f}')

# Overwrite the saved segments with the clipped version
X = X_clipped
np.savez_compressed(str(SEG_DIR / 'segments_all.npz'), X=X, y=y, subjects=subjects)
print('Clipped segments re-saved.')

Before clipping: min=-0.193293  max=0.044531
After clipping : min=-0.000033  max=0.000034
Clipped segments re-saved.


In [9]:
d = np.load(str(SEG_DIR / 'segments_all.npz'))
X_v, y_v = d['X'], d['y']
print(f'Shape: {X_v.shape}')
print(f'PD: {int(y_v.sum())}  Control: {int((y_v==0).sum())}')
print(f'Value range: min={X_v.min():.6f}  max={X_v.max():.6f}')

Shape: (3915, 60, 3000)
PD: 2549  Control: 1366
Value range: min=-0.000033  max=0.000034


In [10]:
# Locate exactly which subject/channel/window produces the extreme values
flat_min_idx = np.unravel_index(np.argmin(X), X.shape)
flat_max_idx = np.unravel_index(np.argmax(X), X.shape)

print(f'Min value {X.min():.6f} V at: window={flat_min_idx[0]}, channel_idx={flat_min_idx[1]}, sample={flat_min_idx[2]}')
print(f'  Subject: {subjects[flat_min_idx[0]]}  Label: {"PD" if y[flat_min_idx[0]]==1 else "Control"}')
print(f'  Channel name: {CANONICAL_CHANNELS[flat_min_idx[1]]}')

print(f'\nMax value {X.max():.6f} V at: window={flat_max_idx[0]}, channel_idx={flat_max_idx[1]}, sample={flat_max_idx[2]}')
print(f'  Subject: {subjects[flat_max_idx[0]]}  Label: {"PD" if y[flat_max_idx[0]]==1 else "Control"}')
print(f'  Channel name: {CANONICAL_CHANNELS[flat_max_idx[1]]}')

# How widespread is this? Check how many windows have ANY value beyond a physiologically
# plausible EEG range (roughly ±0.001V = ±1000µV, generous upper bound even with artifacts)
PLAUSIBLE_LIMIT = 0.001
extreme_windows = np.where(np.abs(X).max(axis=(1,2)) > PLAUSIBLE_LIMIT)[0]
print(f'\nWindows with any value beyond ±{PLAUSIBLE_LIMIT}V: {len(extreme_windows)} / {len(X)} '
      f'({100*len(extreme_windows)/len(X):.1f}%)')

extreme_subjects = np.unique(subjects[extreme_windows])
print(f'Affected subjects: {len(extreme_subjects)} / {len(np.unique(subjects))}')
print(f'Subject list: {list(extreme_subjects)[:20]}{"..." if len(extreme_subjects) > 20 else ""}')

Min value -0.000033 V at: window=101, channel_idx=30, sample=1774
  Subject: 3  Label: PD
  Channel name: AF7

Max value 0.000034 V at: window=101, channel_idx=30, sample=1948
  Subject: 3  Label: PD
  Channel name: AF7

Windows with any value beyond ±0.001V: 0 / 3915 (0.0%)
Affected subjects: 0 / 149
Subject list: []
